# Machine Unlearning — Kaggle Runner
### Master's Research: Machine Unlearning for Multi-Class Image Classification
**Author:** Mikołaj Hajder 264478

> 🔗 **Bookmark this notebook** in your Kaggle account to reopen it each session.

This notebook is the **entry point for all Kaggle experiments**.  
It handles:
1. Setting up working directories (`/kaggle/working/`)
2. Cloning / pulling the latest code from GitHub
3. Installing dependencies
4. Running training and unlearning scripts via shell cells

**Experiments available:**
- `train.py` — standard ResNet-18 training
- `unlearn_naive.py` — naive machine unlearning (full retrain on retain set)
- `train_sisa.py` — SISA ensemble training
- `unlearn_sisa.py` — SISA-based machine unlearning

> **Workflow:** run the setup cells **once per session**, then jump straight to the experiment cell you need.  
> After training, click **Save Version** (top right) to persist checkpoints in the Output tab.


## 1. Set Up Working Directories
Checkpoints and data are stored in `/kaggle/working/`. After each run, click **Save Version** (top right) to commit outputs so they persist across sessions.


In [ ]:
import os

# Kaggle persistent output directory
CKPT_DIR = '/kaggle/working/checkpoints'
DATA_DIR  = '/kaggle/working/data'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR,  exist_ok=True)
print(f'Checkpoints → {CKPT_DIR}')
print(f'Data        → {DATA_DIR}')


## 2. Clone / Update the Repository

In [ ]:
import os

REPO_URL = 'https://github.com/okejka1/master_thesis.git'
REPO_DIR = '/kaggle/working/master_thesis'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
!git log --oneline -3


## 3. Install Dependencies

In [ ]:
%cd /kaggle/working/master_thesis
!pip install -q -r requirements.txt
print('Dependencies installed.')


## 4. Verify Setup

In [ ]:
%cd /kaggle/working/master_thesis
import sys
sys.path.append('/kaggle/working/master_thesis')

import torch, torchvision, yaml
from models import build_resnet18
from utils import STATS, evaluate, set_seed

print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
print('All imports OK.')


---
## 5. Run Experiments

Each cell below runs one experiment. They are **independent** — you can run them in any order after setup.

> **Tip:** The `--checkpoint-dir` flag saves all `.pth` files
> so if the session dies mid-training, the last saved checkpoint is safe.


### 5a. Train — CIFAR-10

In [ ]:
%cd /kaggle/working/master_thesis
!python train.py \
    --config configs/cifar10.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR}


### 5b. Train — CIFAR-100

In [ ]:
%cd /kaggle/working/master_thesis
!python train.py \
    --config configs/cifar100.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR}


### 5c. Naive Unlearning — CIFAR-10 (random forget, 1% of train set)

In [ ]:
%cd /kaggle/working/master_thesis
!python unlearn_naive.py \
    --config configs/cifar10.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR}


### 5d. Naive Unlearning — CIFAR-10 (class-wise forget, e.g. class 0 = 'airplane')

In [ ]:
%cd /kaggle/working/master_thesis
!python unlearn_naive.py \
    --config configs/cifar10.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR} \
    --forget-strategy class \
    --forget-class 0


### 5e. SISA Training — CIFAR-10

In [ ]:
%cd /kaggle/working/master_thesis
# Trains S=5 shards with R=2 slices each (good default for CIFAR-10)
# Results are saved under {CKPT_DIR}/sisa_cifar10/
!python train_sisa.py \
    --config configs/cifar10.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR}


### 5f. SISA Training — CIFAR-100

In [ ]:
%cd /kaggle/working/master_thesis
# Trains S=10 shards with R=5 slices each (good default for CIFAR-100)
# Results are saved under {CKPT_DIR}/sisa_cifar100/
!python train_sisa.py \
    --config configs/cifar100.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR}


### 5g. SISA Unlearning — CIFAR-10 (random forget, 1% of train set)

In [ ]:
%cd /kaggle/working/master_thesis
# Must run 5e first.
# Unlearn results saved to {CKPT_DIR}/sisa_cifar10/unlearn_results.json
!python unlearn_sisa.py \
    --config configs/cifar10.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR}


### 5h. SISA Unlearning — CIFAR-10 (class-wise forget, e.g. class 0 = 'airplane')

In [ ]:
%cd /kaggle/working/master_thesis
# Must run 5e first.
!python unlearn_sisa.py \
    --config configs/cifar10.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR} \
    --forget-strategy class \
    --forget-class 0


### 5i. SISA Unlearning — CIFAR-100 (random forget, 1% of train set)

In [ ]:
%cd /kaggle/working/master_thesis
# Must run 5f first.
!python unlearn_sisa.py \
    --config configs/cifar100.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR}


## 6. Download Checkpoints (optional)

Files saved to `/kaggle/working/` appear in the **Output tab** on the right panel.  
Click the download icon next to any `.pth` file — no extra code needed.


In [ ]:
import os

ckpts = [f for f in os.listdir(CKPT_DIR) if f.endswith('.pth')]
print('Available checkpoints (top-level):')
for c in sorted(ckpts):
    size_mb = os.path.getsize(os.path.join(CKPT_DIR, c)) / 1e6
    print(f'  {c}  ({size_mb:.1f} MB)')

# Also list SISA directories
for tag in ('sisa_cifar10', 'sisa_cifar100'):
    sisa_dir = os.path.join(CKPT_DIR, tag)
    if os.path.isdir(sisa_dir):
        print(f'\nSISA checkpoints ({tag}):')
        for root, dirs, files in os.walk(sisa_dir):
            for f in sorted(files):
                fp = os.path.join(root, f)
                size_mb = os.path.getsize(fp) / 1e6
                rel = os.path.relpath(fp, CKPT_DIR)
                print(f'  {rel}  ({size_mb:.1f} MB)')

# Files in /kaggle/working/ are downloadable from the
# 'Output' tab on the right panel — no extra code needed.
